# Technische Prüfung: Bestand, Restvolumen, Datenlage

Das Gegenstück zu `01_dashboard.ipynb`. Dort steht, was Fachexperten sehen sollen;
hier steht, was zu prüfen ist, bevor man den Zahlen traut – und welche fachlichen
Fragen offen sind.

In [ ]:
#@title Umgebung einrichten und Dashboard laden

import importlib.util

if (
    importlib.util.find_spec("google") is not None
    and importlib.util.find_spec("google.colab") is not None
):
    PAKET_REF = "main"
    PAKET_URL = f"git+https://github.com/it-agile/umsatzprognose-clockodo.git@{PAKET_REF}"
    SETUP_URL = (
        "https://raw.githubusercontent.com/it-agile/umsatzprognose-clockodo/"
        f"{PAKET_REF}/notebooks/setup.py"
    )
    !pip install --quiet "$PAKET_URL"
    !pip install --quiet --force-reinstall --no-deps "$PAKET_URL"
    !curl -sL "$SETUP_URL" -o setup.py

from datetime import date

import setup

stichtag = date.today()
horizont_monate = 3
auslastung_monate = 12
projekte_ohne_auftragsvolumen = 20

dashboard = setup.dashboard(
    stichtag=stichtag,
    horizont_monate=horizont_monate,
    auslastung_monate=auslastung_monate,
)
bestand = dashboard.bestand

print(dashboard.ladebericht())

In [ ]:
#@title Simulation ausführen

dashboard.simuliere(monate=horizont_monate)

## Auftragsvolumen und Restvolumen

`roh` ist `Budget − Verbrauch` und kann negativ sein, weil `budget.hard` in dieser
Installation `false` ist. Prognosewirksam ist `max(0, roh)`: eine Überschreitung kann
nur historisch entstehen, die Prognose überschreitet das Budget nicht.

In [ ]:
#@title Auftragsvolumen und Restvolumen

from umsatzprognose.domaene.zahlen import euro

roh = sum(p.restvolumen_roh or 0.0 for p in bestand.im_prognose_scope)
gesamt = bestand.restvolumen_prognosewirksam

print(f"Auftragsvolumen im Scope:    {euro(bestand.auftragsvolumen):>18}")
print(f"Restvolumen roh:             {euro(roh):>18}")
print(f"Restvolumen prognosewirksam: {euro(gesamt):>18}")
print(f"Summe der Überschreitungen:  {euro(gesamt - roh):>18}")

dashboard.projekttabelle(top=projekte_ohne_auftragsvolumen)

## Aufteilungsschlüssel je Person

Der historische Anteil je Person an den Gesamtstunden des Projekts, aus der
Doppelgruppierung `projects_id` × `users_id`. Er wird unverändert in die Zukunft
fortgeschrieben.

In [ ]:
#@title Aufteilungsschlüssel je Person

for projekt in bestand.im_prognose_scope:
    anteile = projekt.anteil_je_mitarbeiter()
    groesste = sorted(anteile.items(), key=lambda paar: paar[1], reverse=True)
    verteilung = ", ".join(f"{person} {anteil:.0%}" for person, anteil in groesste)
    wort = "Person " if len(anteile) == 1 else "Personen"
    print(f"{projekt.bezeichnung[:48]:<48} {len(anteile):>2} {wort} | {verteilung}")

## Sollarbeitszeit

In [ ]:
#@title Sollarbeitszeit

from umsatzprognose.domaene.zahlen import stunden as stunden_text

aktive = [m for m in bestand.mitarbeiter if m.aktiv]
ohne_sollzeit = [m for m in aktive if m.wochenstunden(bestand.stichtag) is None]
stunden = sum(m.wochenstunden(bestand.stichtag) or 0 for m in aktive)

print(f"Aktive Personen: {len(aktive)}, ohne hinterlegte Sollzeit: {len(ohne_sollzeit)}")
print(f"Vereinbarte Wochenstunden gesamt: {stunden_text(stunden)}")

## Kapazität

Wer über die letzten `auslastung_monate` abgeschlossenen Monate am meisten Kapazität
hatte - der laufende (Stichtags-)Monat ist unvollständig und bleibt deshalb außen vor,
genau wie bei `Dashboard.auslastung_je_mitarbeiter()`. Dazu, wie sich die in der
Simulation über den Prognosehorizont tatsächlich verbrauchte Kapazität auf die
Projekte im Scope verteilt - in Personentagen à 7 Stunden.

In [ ]:
#@title Kapazität je Mitarbeiter

dashboard.kapazitaet_je_mitarbeiter(top=100)

In [ ]:
#@title Kapazität je Projekt

dashboard.kapazitaet_je_projekt(top=1_000)

## Datenlage

In [ ]:
#@title Datenlage: Hinweise

dashboard.hinweise(max_anzahl_betroffen=1_000)

### Aktive Projekte ohne Budget

Sie fallen aus der Prognose, weil ihnen ein bezifferbares Auftragsvolumen fehlt. Ein
Blick auf die Liste unten zeigt, was das überwiegend ist: Schulungs- und
Ausbildungsprodukte aus dem offenen Kursangebot, also Katalogpositionen ohne
beauftragtes Volumen. Zu prüfen bleibt, ob darunter
echte Bestandsprojekte stecken, bei denen nur das Budget fehlt.

In [ ]:
#@title Aktive Projekte ohne Budget

# filter kann hier direkt angepasst werden
projekt_filter = [
    "it-agile GmbH",
    "Öffentliche Schulung",
]

dashboard.projekte_ohne_budget(filter=projekt_filter)

### Aktive Projekte, die als abgeschlossen markiert sind

Sie fallen aus der Prognose. Bitte prüfen!

In [ ]:
#@title Aktive Projekte, die als abgeschlossen markiert sind

beendet = [p for p in bestand.aktive_projekte if p.abgeschlossen]
for projekt in beendet:
    offen = projekt.restvolumen_prognosewirksam
    betrag = "kein Budget" if offen is None else euro(offen)
    print(f"  {projekt.bezeichnung[:58]:<58} offen: {betrag}")